# Homomorphic Time-Lock Puzzles
### Walk-through of Malavolta & Thyagarajan ([eprint 2019/635](https://eprint.iacr.org/2019/635)) and demo

This notebook is a **implementation** of the two HTLP schemes from the paper:

| Scheme | Section | Homomorphism | We use it for |
|---|---|---|---|
| **LHTLP** | §4.1 | linear: sum of secrets mod N | E-voting tally (`/vote`) |
| **MHTLP** | §4.2 | multiplicative; with ±1 encoding ⇒ XOR | Multi-party coin flip (`/flip`) |

<!-- Both schemes share the same skeleton:

1. **`PSetup`** — pick a strong RSA modulus `N` and a generator `g` of the cyclic subgroup `J_N ⊂ Z*_N`. Pre-compute `h = g^(2^T) mod N` using the secret trapdoor `φ(N)/2`.
2. **`PGen(s)`** — produce a puzzle `Z = (u, v)` that hides `s` for time `~T`.
3. **`PEval(Z₁,…,Zₙ)`** — combine puzzles *without* solving them. The output is a fresh puzzle encoding `f(s₁,…,sₙ)`.
4. **`PSolve(Z)`** — perform `T` *inherently sequential* squarings to undo the blinding and recover `s`.

The magic ingredient is the **sequential-squaring assumption**: computing `x^(2^T) mod N` cannot be parallelised — without knowing `φ(N)`, you have to do `T` squarings one after another. This is what gives time-lock puzzles their "time" property.

The **homomorphism** is what makes them special compared to ordinary time-lock puzzles: instead of solving N puzzles to tally N votes, the tallier homomorphically combines all puzzles into **one** puzzle and solves *that*. Cost is independent of the number of voters.

We will build each scheme one operation at a time — every section is a short paragraph followed by a code block you can step through. -->


<!-- ## 1. Number-theoretic background

A few definitions used throughout the paper

### 1.1 Strong RSA integer
`N = p · q` where both `p = 2p'+1` and `q = 2q'+1` are **safe primes** (i.e. `p', q'` are also prime).

- Euler totient: `φ(N) = (p-1)(q-1) = 4·p'·q'`.
- The group of "elements with Jacobi symbol +1" — called `J_N` — is **cyclic of order `φ(N)/2 = 2·p'·q'`** when `N` is a strong RSA integer.
- A generator of `J_N` can be sampled cheaply: pick `g̃ ∈ Z*_N` uniformly and set `g := -g̃² mod N`. With overwhelming probability the order of `g̃` is `φ(N)/2` or `φ(N)/4`, so `-g̃²` lands in `J_N` and generates it.

### 1.2 Why `J_N` and not all of `Z*_N`?
If we worked in `Z*_N`, an adversary could distinguish `x^(2^T)` from random by computing the Jacobi symbol (efficient *without* the factorisation). Restricting to `J_N` blocks that trivial attack.

### 1.3 Sequential-squaring assumption (RSW96 / §2.1)
> Given `(N, g, T, x)` with `x ∈ J_N`, no polynomial-size circuit of depth `< T` can distinguish `x^(2^T)` from a uniform element of `J_N`.

Knowing `φ(N)/2` lets the setup author short-circuit by reducing `2^T mod φ(N)/2` first — but **solvers don't know `φ(N)`**, so they must perform `T` squarings in sequence.

### 1.4 Paillier's `(1+N)^s mod N²` trick
By the binomial theorem:
$$(1+N)^s \;=\; 1 + sN + \binom{s}{2}N^2 + \dots \;\equiv\; 1 + sN \pmod{N^2}.$$
So once we recover `(1+N)^s mod N²`, the secret `s` is just `((1+N)^s − 1) / N`. This is the "easy discrete log" Paillier exploits, and it's why LHTLP can embed `s ∈ Z_N` into `N²`-arithmetic and pull it back out cheaply. -->


## 1. Common helpers

Two utility functions that both schemes will use later.

### 1.1 `generate_strong_prime` — safe-prime sampling

A safe prime is `p = 2·p' + 1` with `p'` also prime.

<!-- * **Strong RSA structure.** `N = p·q` with both safe is a *strong RSA integer*, so `J_N` is cyclic of known order `φ(N)/2`. The security proofs in §4 rely on this. -->
<!-- * **Blum integer for free.** Safe primes always satisfy `p ≡ 3 (mod 4)`, which the MHTLP/XOR variant requires. -->

In [23]:
import random
import time
from sympy import isprime, randprime


def generate_strong_prime(bits: int):
    '''Sample a safe prime p = 2·p' + 1 (p' also prime). Returns (p, p').'''
    while True:
        p_prime = randprime(2 ** (bits - 2), 2 ** (bits - 1))
        p = 2 * p_prime + 1
        if isprime(p):
            return p, p_prime


# Quick sanity check: a 32-bit safe prime.
p, p_prime = generate_strong_prime(32)
print(f'p  = {p}')
print(f"p' = (p-1)/2 = {p_prime} prime? {isprime(p_prime)}")

p  = 3889512503
p' = (p-1)/2 = 1944756251 prime? True


### 1.2 `calibrate_T` — picking `T` for a wall-clock target

This is *not* part of the cryptographic scheme. Just measures how many modular squarings/sec the CPU manages, then returns `T = squarings_per_sec · target_seconds`.

In [24]:
def calibrate_T(target_seconds: float, bits: int = 128) -> int:
    '''Estimate squarings/sec, then scale to T = sq/sec · target_seconds.'''
    test_N = generate_strong_prime(bits)[0] * generate_strong_prime(bits)[0]
    x = random.randrange(2, test_N)
    squarings, start = 0, time.time()
    while time.time() - start < 1.0:
        x = pow(x, 2, test_N)        # one squaring = one tick of the clock
        squarings += 1
    return int(squarings * target_seconds)


squarings_per_sec = calibrate_T(1)
print(f'This CPU does about {squarings_per_sec:,} squarings/sec.')
print(f'For ~30 s of solving: T ≈ {30 * squarings_per_sec:,}.')

This CPU does about 859,154 squarings/sec.
For ~30 s of solving: T ≈ 25,774,620.


## 2. The Linearly-Homomorphic Time-Lock Puzzle

The data flow we are building:

```
PSetup ────► public params  pp = (N, g, h, T)
                │
                ▼
 secret s  ──► PGen ────► puzzle Z = (u, v)
                                │
                                ▼
        many puzzles  ────► PEval ────► combined puzzle
                                              │
                                              ▼
                                            PSolve  ────► f(s₁,…,sₙ)
```


### 2.1 `PSetup` (public parameters)

The trusted authority generates the public parameters once:

<!-- 1. Sample two safe primes `p, q` and set `N := p·q`.
2. Sample `g̃ ←$ Z*_N`; set `g := −g̃² mod N`. With overwhelming probability `g` generates `J_N`.
3. Compute `h := g^(2^T) mod N`.
   * **With the trapdoor `order(J_N) = φ(N)/2`**, reduce `2^T mod φ(N)/2` first — one fast exponentiation.
   * **Without the trapdoor**, the only way is `T` sequential squarings — that's what the time-lock relies on. -->

In [25]:
def lhtlp_setup(bits: int, T: int):
    '''Generate public parameters pp = (N, g, h, T) for LHTLP.

    Returns a dict.'''
    # Step 1: two safe primes → strong RSA modulus.
    p, _ = generate_strong_prime(bits)
    q, _ = generate_strong_prime(bits)
    N = p * q
    N2 = N * N

    # The trapdoor: order of J_N = φ(N)/2 = 2·p'·q'. Kept secret by setup author.
    order_JN = ((p - 1) * (q - 1)) // 2

    # Step 2: create a generator of J_N
    while True:
        g_tilde = random.randrange(2, N)
        g = (-pow(g_tilde, 2, N)) % N
        if g > 1:
            break

    # Step 3: fast h = g^(2^T) mod N using the trapdoor.
    # Anyone without order_JN must do T squarings instead — that's the time-lock.
    h = pow(g, pow(2, T, order_JN), N)

    return {'N': N, 'N2': N2, 'g': g, 'h': h, 'T': T}


# Tiny parameters example:
pp = lhtlp_setup(bits=16, T=10)
print(f"N  = {pp['N']}   ({pp['N'].bit_length()} bits)")
print(f"g  = {pp['g']}")
print(f"h  = {pp['h']}   (= g^(2^T) mod N)")
print(f"T  = {pp['T']}")

N  = 1409711521   (31 bits)
g  = 956920413
h  = 715547150   (= g^(2^T) mod N)
T  = 10


### 2.2 `PGen` (secret encryption)

Given `s ∈ Z_N` and public params `pp`:

1. Sample fresh randomness `r ←$ {1, …, N²}`.
   <!-- *(Sampling here, rather than from `{1, …, φ(N)/2}` which we don't know, is statistically fine by Lemma 1 in §2.)* -->
2. `u := g^r mod N`.
3. `v := h^(r·N) · (1+N)^s mod N²`.

<!-- The factor `h^(rN)` is a one-time blinding pad; the factor `(1+N)^s` carries the secret in the Paillier-style embedding. -->

In [26]:
def lhtlp_pgen(pp, s: int):
    '''Encrypt secret s into a puzzle Z = (u, v).'''
    N, N2, g, h = pp['N'], pp['N2'], pp['g'], pp['h']

    r = random.randrange(1, N2)            # per-puzzle randomness - fresh random exponent
    u = pow(g, r, N)                       # u = g^r mod N         - PSolve squares this T times
    blinding   = pow(h, r * N, N2)         # h^(r·N) mod N²        - hides (1+N)^s
    payload    = pow(1 + N, s, N2)         # (1+N)^s mod N²        - encoded secret
    v = (blinding * payload) % N2          # v = h^(rN) · (1+N)^s  - maskes payload

    return (u, v)                          # puzzle publication


Z1 = lhtlp_pgen(pp, 5)
Z2 = lhtlp_pgen(pp, 10)
print(f'Z1 (s=5)  = (u={Z1[0]}, v={Z1[1]})')
print(f'Z2 (s=10) = (u={Z2[0]}, v={Z2[1]})')

Z1 (s=5)  = (u=1129366424, v=594969659570318347)
Z2 (s=10) = (u=539719287, v=996442688415749473)


### 2.3 `PEval` (summing puzzles homomorphically)
<!-- 
Given `Z_i = (u_i, v_i)` encoding `s_i`, define
$$\tilde u := \prod_i u_i \bmod N, \qquad \tilde v := \prod_i v_i \bmod N^2.$$
Plug in the definitions of `u_i, v_i`:
$$\tilde u = g^{\sum r_i}, \qquad \tilde v = h^{N\sum r_i}\cdot(1+N)^{\sum s_i}.$$
That's exactly a fresh LHTLP puzzle for `Σ s_i` with randomness `Σ r_i`. Crucially the runtime of `PEval` is **independent of `T`**. -->

In [27]:
# puzzles = 
def lhtlp_peval(pp, puzzles):
    '''Combine puzzles homomorphically. Output is a fresh puzzle for Σ s_i.'''
    N, N2 = pp['N'], pp['N2']
    u_tilde, v_tilde = 1, 1
    for u, v in puzzles: # (u = randomness, v = masked secret)
        u_tilde = (u_tilde * u) % N
        v_tilde = (v_tilde * v) % N2
    return (u_tilde, v_tilde) # return of combined puzzle


Z_sum = lhtlp_peval(pp, [Z1, Z2])
print(f'Combined puzzle for s = 5 + 10 :  {Z_sum}')

Combined puzzle for s = 5 + 10 :  (143064595, 212446605533143727)


### 2.4 `PSolve` — the slow step

Three sub-steps:

1. **`w := u^(2^T) mod N`** — the *only* slow part. `T` sequential squarings; no shortcut without `φ(N)`.
2. **Undo the blinding.** Note `w = (g^r)^(2^T) = (g^(2^T))^r = h^r`, so `w^N = h^(rN)`. Compute `(v · (w^N)^{-1}) mod N²` to expose `(1+N)^s mod N²`.
3. **Paillier discrete log.** Because `(1+N)^s ≡ 1 + sN mod N²`, the secret is just `s = (blob − 1) / N`.

In [28]:
def lhtlp_psolve(pp, puzzle):
    '''Recover the secret from a puzzle. This is the slow op (T squarings).'''
    N, N2, T = pp['N'], pp['N2'], pp['T']
    u, v = puzzle

    # Step 1: T sequential squarings.
    w = u
    for _ in range(T):
        w = pow(w, 2, N)

    # Step 2: divide v by w^N inside Z*_{N²}.
    wN = pow(w, N, N2)           # reconstructs blinding pad of PGen
    wN_inv = pow(wN, -1, N2)     # mod inverse
    blob = (v * wN_inv) % N2     # equals (1+N)^s mod N² -> mult v by mod inv

    # Step 3: discrete log.
    return (blob - 1) // N


recovered = lhtlp_psolve(pp, Z_sum)
print(f'PSolve(Z_sum) = {recovered}   ← expected 15')
assert recovered == 15

PSolve(Z_sum) = 15   ← expected 15


### 2.5 `PSolve` in detail

Same call as above, but printing every intermediatd.

In [29]:
u, v = Z_sum                            # unpacking the combined puzzle: u commits to randomness, v hides the secret
N, N2, T = pp['N'], pp['N2'], pp['T']   # grabing the moduli and the time-lock parameter

# Step 1 — redo the time-lock: square u exactly T times to reach h^r.
w = u
for _ in range(T):                      # what actually takes time
    w = pow(w, 2, N)                    # w ← w² mod N
print(f'w = u^(2^T) mod N  = {w}')

# Step 2 — rebuild the blinding pad and invert it so we can cancel it out of v.
wN = pow(w, N, N2)
wN_inv = pow(wN, -1, N2)                # mod inverse
print(f'w^N mod N²         = {wN}')

# Step 3 — divide v by the pad → out pops (1+N)^s
blob = (v * wN_inv) % N2
direct = pow(1 + N, 15, N2)             # Sanity check
print(f'(v / w^N) mod N²   = {blob}')
print(f'(1+N)^15 mod N²    = {direct}')
assert blob == direct

s = (blob - 1) // N
print(f's = (blob − 1)/N   = {s}')      # reovered secret
assert s == 15

w = u^(2^T) mod N  = 837409942
w^N mod N²         = 1383974932814290028
(v / w^N) mod N²   = 21145672816
(1+N)^15 mod N²    = 21145672816
s = (blob − 1)/N   = 15


### 2.6 Four functions into a class

The demo (`Demo/htlp.py`) exposes LHTLP as a class so it can hold the public params as state. The class is just a thin wrapper around the four functions above

In [30]:
class LHTLP:
    '''Thin wrapper matching the API used in Demo/htlp.py.'''

    def __init__(self, bits: int, T: int):
        self.pp = lhtlp_setup(bits, T)

    # Expose the parts the demo's other layers expect:
    @property
    def N(self):  return self.pp['N']
    @property
    def T(self):  return self.pp['T']

    def PGen(self, s):           return lhtlp_pgen(self.pp, s)
    def PEval(self, puzzles):    return lhtlp_peval(self.pp, puzzles)
    def PSolve(self, puzzle):    return lhtlp_psolve(self.pp, puzzle)


# Quick check the wrapper agrees with the functional version.
toy = LHTLP(bits=16, T=10)
ZA = toy.PGen(7)
ZB = toy.PGen(35)
assert toy.PSolve(toy.PEval([ZA, ZB])) == 42
print('class-wrapped LHTLP works')

class-wrapped LHTLP works


## 3. The Multiplicatively-Homomorphic Time-Lock Puzzle (§4.2)

<!-- Same skeleton as LHTLP, smaller secret space, simpler arithmetic.

| | LHTLP | MHTLP |
|---|---|---|
| Secret space | `Z_N` | `J_N` (we use `{+1, −1}`) |
| `v` lives in | `Z*_{N²}` | `Z*_N` |
| `PGen` | `v := h^(r·N)·(1+N)^s` | `v := h^r · s` |
| `PEval` | `∏ v_i mod N²` | `∏ v_i mod N` |
| `PSolve` | `(v/w^N − 1)/N` | `v / w mod N` |
| Homomorphism | additive over secrets | multiplicative over secrets |

### XOR via ±1 encoding (§4.2, "XOR-Homomorphism" paragraph)
When `N` is a **Blum integer** (`p ≡ q ≡ 3 mod 4`), `{+1, −1}` is a subgroup of `J_N`. Encode `b ∈ {0,1}` as `(−1)^b ∈ {+1, N−1}`. Then multiplication = XOR:

| `b₁` | `b₂` | `(-1)^{b₁}·(-1)^{b₂}` | `b₁ ⊕ b₂` |
|---|---|---|---|
| 0 | 0 | +1 | 0 |
| 0 | 1 | −1 | 1 |
| 1 | 1 | +1 | 0 |

So a combined MHTLP puzzle encodes the XOR of every bit that went in. -->

### 3.1 `PSetup` (identical to LHTLP)

<!-- Same `(N, g, h, T)`, no `N²` needed. Returning a dict with the same shape minus `N2` keeps the rest of the code uniform. -->

In [31]:
def mhtlp_setup(bits: int, T: int):
    p, _ = generate_strong_prime(bits)
    q, _ = generate_strong_prime(bits)
    N = p * q
    order_JN = ((p - 1) * (q - 1)) // 2
    while True:
        g_tilde = random.randrange(2, N)
        g = (-pow(g_tilde, 2, N)) % N
        if g > 1:
            break
    h = pow(g, pow(2, T, order_JN), N)
    return {'N': N, 'g': g, 'h': h, 'T': T}


pp_m = mhtlp_setup(bits=16, T=10)
print(f"N  = {pp_m['N']}   ({pp_m['N'].bit_length()} bits)")
print(f"g  = {pp_m['g']}")
print(f"h  = {pp_m['h']}   (= g^(2^T) mod N)")
print(f"T  = {pp_m['T']}")

N  = 3916962781   (32 bits)
g  = 656466403
h  = 146248946   (= g^(2^T) mod N)
T  = 10


### 3.2 `PGen`

<!-- `b ∈ {0,1}` → `s ∈ {+1, N−1}`; then `u := g^r mod N`, `v := h^r · s mod N`. Notice `v` lives in `Z*_N`, not `Z*_{N²}` — no Paillier embedding needed because the secret is already inside the group. -->

In [32]:
def mhtlp_pgen(pp, bit: int):             
    assert bit in (0, 1)                   # MHTLP only encodes a single bit per puzzle
    N, g, h = pp['N'], pp['g'], pp['h']

    s = 1 if bit == 0 else N - 1           # encode bit as ±1: 0 → +1, 1 → −1
    r = random.randrange(1, N * N)         # same statistical-distance argument as LHTLP
    u = pow(g, r, N)                       # commitement to r
    v = (pow(h, r, N) * s) % N             # secret masked by blinding pad h^r
    return (u, v)


for b in (0, 1):
    u, v = mhtlp_pgen(pp_m, b)             # build one puzzle per bit value
    encoded = '+1 (tails)' if b == 0 else 'N-1 ≡ -1 (heads)'
    print(f'bit={b}  s={encoded:18s}  puzzle u={u}, v={v}')

bit=0  s=+1 (tails)          puzzle u=3905069803, v=2189823667
bit=1  s=N-1 ≡ -1 (heads)    puzzle u=2537215410, v=1692620622


### 3.3 `PEval` (multiply puzzles = XOR the bits)

Identical structure to LHTLP's PEval, but everything is mod `N` (no `N²`). The combined puzzle encodes the *product* of the secrets — which, with our ±1 encoding, is the XOR of the bits.

In [33]:
def mhtlp_peval(pp, puzzles):
    N = pp['N']
    u_tilde, v_tilde = 1, 1
    for u, v in puzzles:
        u_tilde = (u_tilde * u) % N
        v_tilde = (v_tilde * v) % N
    return (u_tilde, v_tilde)


bits = [1, 0, 1, 1] # example bits
puzzles = [mhtlp_pgen(pp_m, b) for b in bits]
combined = mhtlp_peval(pp_m, puzzles)
print(f'bits = {bits}   →   combined puzzle = {combined}')

bits = [1, 0, 1, 1]   →   combined puzzle = (2300091316, 1147863665)


### 3.4 `PSolve` (XOR recovery)

<!-- 1. `w := u^(2^T) mod N` — the slow step, same as LHTLP.
2. `s := v · w^{-1} mod N` — the secret, which is `±1`.
3. Map back to a bit: `+1 → 0`, `N−1 → 1`. -->

In [34]:
def mhtlp_psolve(pp, puzzle):
    N, T = pp['N'], pp['T']
    u, v = puzzle

    # T sequential squarings, slow time-lock step.
    w = u
    for _ in range(T):          # T squarings, inherently sequential
        w = pow(w, 2, N)        # after the loop: w = u^(2^T) = h^r mod N

    # Undo the blinding, for MHTLP this is just modular division.
    s = (v * pow(w, -1, N)) % N
    return 0 if s == 1 else 1


outcome  = mhtlp_psolve(pp_m, combined)
expected = bits[0] ^ bits[1] ^ bits[2] ^ bits[3] # groound truth
print(f'XOR ground truth  = {expected}')
print(f'PSolve gives      = {outcome}')
assert outcome == expected

XOR ground truth  = 1
PSolve gives      = 1


### 3.5 Wrap into the class the demo uses

In [35]:
class MHTLP:
    def __init__(self, bits: int, T: int):
        self.pp = mhtlp_setup(bits, T)

    @property
    def N(self):  return self.pp['N']
    @property
    def T(self):  return self.pp['T']

    def PGen(self, bit):         return mhtlp_pgen(self.pp, bit)
    def PEval(self, puzzles):    return mhtlp_peval(self.pp, puzzles)
    def PSolve(self, puzzle):    return mhtlp_psolve(self.pp, puzzle)


m = MHTLP(bits=16, T=10)
assert m.PSolve(m.PEval([m.PGen(1), m.PGen(1), m.PGen(0)])) == 0   # 1⊕1⊕0
print('class-wrapped MHTLP works ✓')

class-wrapped MHTLP works ✓


## 4. Application 1: e-voting (Codex vs Claude)

This mirrors the `/vote` flow in the live demo.

### Protocol (paper §6.1, specialised to `m = 2` candidates)
**Setup (trusted authority, one-time):** publish `pp` from `lhtlp_setup`.

**Voting phase.** Each voter `Vᵢ` picks a candidate and posts a 2-tuple of LHTLP puzzles, e.g. `(PGen(1), PGen(0))` for Codex.

**Counting phase.** The tallier `PEval`s each column and runs `PSolve` **twice** — work is independent of the number of voters.

<!-- ### What the live demo does differently
The browser only generates **one** puzzle (the one for the chosen candidate, with secret = 1); the server fills the matching `PGen(0)`. Cryptographically identical, but a serious deployment would have the voter generate *both* puzzles client-side and attach a NIZK proving exactly one encodes 1 (paper p.4 footnote 3). -->

### 4.1 Set up the election and cast ballots

In [36]:
election = LHTLP(bits=64, T=200)   # small params so the notebook is snappy

voter_choices = ['Codex', 'Codex', 'Claude', 'Codex', 'Claude', 'Codex', 'Codex']

ballots = []
for choice in voter_choices:
    if choice == 'Codex':
        ballots.append((election.PGen(1), election.PGen(0)))   # (codex, claude)
    else:
        ballots.append((election.PGen(0), election.PGen(1)))

print(f'{len(ballots)} ballots cast — each is a pair of LHTLP puzzles.')
print(f'One puzzle component has ~{(election.N * election.N).bit_length()} bits.')

7 ballots cast — each is a pair of LHTLP puzzles.
One puzzle component has ~256 bits.


### 4.2 Tally homomorphically, then solve twice

In [37]:
codex_puzzles  = [b[0] for b in ballots]
claude_puzzles = [b[1] for b in ballots]

t0 = time.time()
codex_combined  = election.PEval(codex_puzzles)
claude_combined = election.PEval(claude_puzzles)
print(f'PEval over {len(ballots)} ballots: {(time.time()-t0)*1000:.2f} ms — independent of T.')

t0 = time.time()
codex_total  = election.PSolve(codex_combined)
claude_total = election.PSolve(claude_combined)
print(f'2× PSolve (T={election.T}): {(time.time()-t0)*1000:.2f} ms.\n')

print(f'Result:  Codex = {codex_total},  Claude = {claude_total}')
print(f'Truth :  Codex = {voter_choices.count("Codex")},  Claude = {voter_choices.count("Claude")}')
assert codex_total == voter_choices.count('Codex')
assert claude_total == voter_choices.count('Claude')

PEval over 7 ballots: 0.55 ms — independent of T.
2× PSolve (T=200): 1.19 ms.

Result:  Codex = 5,  Claude = 2
Truth :  Codex = 5,  Claude = 2


## 5. Application 2: multi-party coin flip

This mirrors the `/flip` flow. The XOR variant noted at the end of §4.2 is what we run, and what's shown below.

**Protocol.**
1. Setup publishes `pp` from `mhtlp_setup`.
2. Each participant `Pᵢ` draws `bᵢ ←$ {0,1}` and broadcasts `Zᵢ = mhtlp_pgen(pp, bᵢ)`.
3. Anyone can combine via `PEval`. Solving once reveals `b = b₁ ⊕ … ⊕ bₙ`.

**Bias resistance.** If `n−1` parties collude, they can choose bits after seeing the others' puzzles, but they cannot see the honest party's bit before committing (it's behind the time-lock). XOR with a fresh uniform bit is uniform, so the coin stays fair.

### 5.1 Five participants, each broadcasts a random time-locked bit

In [38]:
flip = MHTLP(bits=64, T=200)

participants = 5
bits = [random.randint(0, 1) for _ in range(participants)]
puzzles = [flip.PGen(b) for b in bits]

print('Participant bits (hidden behind time-locks):', bits)
print('(In a real run, the bits would not be visible until PSolve completes.)')

Participant bits (hidden behind time-locks): [0, 1, 0, 0, 0]
(In a real run, the bits would not be visible until PSolve completes.)


### 5.2 Combine and solve once

In [39]:
combined = flip.PEval(puzzles)
outcome  = flip.PSolve(combined)

expected = 0
for b in bits:
    expected ^= b

face = 'heads' if outcome == 1 else 'tails'
print(f'Coin flip result : {outcome}  ({face})')
print(f'XOR ground truth : {expected}')
assert outcome == expected

Coin flip result : 1  (heads)
XOR ground truth : 1


### 5.3 Client-side puzzle generation in the live demo

In the `/flip` demo each participant's puzzle is generated entirely inside their **browser tab** — the server never sees the bit. The steps map directly onto §3.2:

| Step | Python (this notebook) | Browser (`htlp.js`) |
|---|---|---|
| public params | `mhtlp_setup` returns `pp` | `GET /flip-params` → `{ N, g, h, T }` |
| random bit | `random.randint(0, 1)` | `crypto.getRandomValues` (browser CSPRNG) |
| puzzle | `mhtlp_pgen(pp, bit)` | `mhtlpPGen(pp, bit)` — same arithmetic, BigInt |
| server receives | — | `POST /flip { u, v }` — bit **not** included |

`PSolve` runs server-side only after the collection phase closes.

<!-- ## 7. Where does each piece live in the demo?

| Notebook concept | Server-side (`Demo/htlp.py`) | Browser-side (`Demo/static/htlp.js`) |
|---|---|---|
| `lhtlp_setup` | `LHTLP.__init__` | (browser receives `(N, g, h, T)` via `/vote-params`) |
| `lhtlp_pgen`  | `LHTLP.PGen` (server tests + filler `PGen(0)`) | `lhtlpPGen` — voter generates the puzzle locally so their bit never leaves the browser in cleartext |
| `lhtlp_peval` | `LHTLP.PEval` (in `/start_tally`) | — |
| `lhtlp_psolve`| `LHTLP.PSolve` w/ `progress_cb` for the UI bar | — |
| `mhtlp_*`     | `MHTLP` class | `mhtlpPGen` — each participant flips a coin locally and posts the puzzle |

Two non-cryptographic conveniences in the demo:

- **`progress_cb`** in `PSolve` ticks a percentage every `T/100` squarings so the frontend can animate a progress bar — it does not affect correctness or security.
- **`calibrate_T`** runs once at app startup to pick a `T` matching the user-chosen wall-clock duration.

## 8. Take-aways

- **The magic of HTLP is `PEval`.** The solver pays cost proportional to `T`, not `n·T`, which is what makes e-voting, sealed-bid auctions, coin flipping, and contract signing actually scale.
- The construction sits on two assumptions: sequential squaring over `J_N` (the "time" part) and DCR/DDH (the "secrecy" part).
- Everything in this notebook is `O(\log)` integer arithmetic — no pairings, no lattices, no obfuscation. That's why the entire server fits in `Demo/htlp.py` (≈130 lines) and the browser client in `Demo/static/htlp.js` (≈60 lines).
- Not implemented here: the fully-homomorphic construction (§4.3 — needs iO), the semi-compact branching-program scheme (§5.1), and the public-coin / reusable setup variants (§5.2–5.3). -->


## 6. Worked examples with tiny numbers

The cells above showed the algebra at production scale. The two cells below redo each application **end-to-end with deliberately small parameters**, printing every intermediate so you can verify by hand that:

- each `PGen` produces a sensible `(u, v)`,
- `PEval`'s combined `(ũ, ṽ)` really is the component-wise product mod the right modulus,
- `PSolve` recovers exactly the secret you'd expect from the homomorphism.

Parameters are kept tiny on purpose (`bits=16`, `T=5`) so every integer fits on one line.

### 6.1 Application 1: e-voting, every intermediate printed
<!-- 
Three voters, two candidates (Codex vs Claude). Vote tally = `[Codex, Claude, Codex]`, so the expected counts are **Codex = 2, Claude = 1**.

For each ballot we print:
- the per-puzzle randomness `r`, the commitment `u = g^r mod N`, and the masked payload `v = h^(rN) · (1+N)^s mod N²`.

Then for each candidate column we print:
- the running products `ũ = ∏ uᵢ mod N` and `ṽ = ∏ vᵢ mod N²` after each fold,
- the `PSolve` chain: `w = u^(2^T)` after each squaring, `w^N mod N²`, the unblinded `blob = (1+N)^S mod N²`, and finally the recovered `S`. -->

In [ ]:
import random

# ---------- 1. Setup (tiny parameters) -------------------------------------
random.seed(0)                                          # deterministic output
demo_pp = lhtlp_setup(bits=16, T=5)                     # T=5 so we can print every squaring
N, N2, g, h, T = demo_pp['N'], demo_pp['N2'], demo_pp['g'], demo_pp['h'], demo_pp['T']
print('--- public parameters -----------------------------------------------')
print(f'N  = {N}   ({N.bit_length()} bits)')
print(f'N² = {N2}')
print(f'g  = {g}')
print(f'h  = g^(2^T) mod N = {h}')
print(f'T  = {T}\n')

# ---------- 2. Three voters cast their ballots -----------------------------
choices  = ['Codex', 'Claude', 'Codex']                 # ground truth: 2 vs 1
expected = (choices.count('Codex'), choices.count('Claude'))

def pgen_verbose(s, label):
    '''lhtlp_pgen but printing every intermediate.'''
    r = random.randrange(1, N2)
    u = pow(g, r, N)
    blinding = pow(h, r * N, N2)
    payload  = pow(1 + N, s, N2)
    v = (blinding * payload) % N2
    print(f'  {label}: s={s}  r={r}')
    print(f'         u = g^r mod N        = {u}')
    print(f'         blinding = h^(rN) %N²= {blinding}')
    print(f'         payload  = (1+N)^s%N²= {payload}')
    print(f'         v = blinding·payload = {v}')
    return (u, v)

ballots = []
for i, choice in enumerate(choices, start=1):
    print(f'--- ballot {i}  (voter chose {choice}) ---------------------------------')
    s_codex, s_claude = (1, 0) if choice == 'Codex' else (0, 1)
    Z_codex  = pgen_verbose(s_codex,  'codex-puzzle ')
    Z_claude = pgen_verbose(s_claude, 'claude-puzzle')
    ballots.append((Z_codex, Z_claude))
    print()

# ---------- 3. Tally each column with PEval --------------------------------
def peval_verbose(puzzles, column_name):
    print(f'--- PEval over the {column_name} column -----------------------------')
    u_tilde, v_tilde = 1, 1
    for k, (u, v) in enumerate(puzzles, start=1):
        u_tilde = (u_tilde * u) % N
        v_tilde = (v_tilde * v) % N2
        print(f'  after ballot {k}:  ũ = {u_tilde}   ṽ = {v_tilde}')
    print(f'  combined puzzle: ({u_tilde}, {v_tilde})\n')
    return (u_tilde, v_tilde)

codex_combined  = peval_verbose([b[0] for b in ballots], 'Codex')
claude_combined = peval_verbose([b[1] for b in ballots], 'Claude')

# ---------- 4. PSolve each combined puzzle, step by step -------------------
def psolve_verbose(puzzle, label):
    print(f'--- PSolve on {label} ----------------------------------')
    u, v = puzzle
    w = u
    print(f'  start: w = u = {w}')
    for step in range(1, T + 1):
        w = pow(w, 2, N)
        print(f'  after squaring #{step}:  w = {w}')
    wN     = pow(w, N, N2)
    wN_inv = pow(wN, -1, N2)
    blob   = (v * wN_inv) % N2
    s = (blob - 1) // N
    print(f'  w^N mod N²        = {wN}')
    print(f'  (w^N)^-1 mod N²   = {wN_inv}')
    print(f'  blob = v·wN_inv   = {blob}   (should equal (1+N)^S mod N²)')
    print(f'  check (1+N)^{s} %N²= {pow(1 + N, s, N2)}')
    print(f'  recovered S       = (blob-1)//N = {s}\n')
    return s

codex_count  = psolve_verbose(codex_combined,  'Codex column')
claude_count = psolve_verbose(claude_combined, 'Claude column')

# ---------- 5. Final tally --------------------------------------------------
print('=== FINAL TALLY =====================================================')
print(f'Codex  = {codex_count}    (expected {expected[0]})')
print(f'Claude = {claude_count}    (expected {expected[1]})')
assert (codex_count, claude_count) == expected
print('tally matches the ground truth')

--- public parameters -----------------------------------------------
N  = 2716597981   (32 bits)
N² = 7379904590373276361
g  = 2421975149
h  = g^(2^T) mod N = 779364023
T  = 5

--- ballot 1  (voter chose Codex) ---------------------------------
  codex-puzzle : s=1  r=373402508605428822
         u = g^r mod N        = 467226597
         blinding = h^(rN) %N²= 2293823403206403556
         payload  = (1+N)^s%N²= 2716597982
         v = blinding·payload = 4597342164463586911
  claude-puzzle: s=0  r=4481891911369680483
         u = g^r mod N        = 1915152334
         blinding = h^(rN) %N²= 655810493037707733
         payload  = (1+N)^s%N²= 1
         v = blinding·payload = 655810493037707733

--- ballot 2  (voter chose Claude) ---------------------------------
  codex-puzzle : s=0  r=3302422582397125125
         u = g^r mod N        = 750585165
         blinding = h^(rN) %N²= 7077067162852041759
         payload  = (1+N)^s%N²= 1
         v = blinding·payload = 7077067162852041759
  cla

### 6.2 Application: coin flip, every intermediate printed
<!-- 
Four participants, each commits to a random bit via MHTLP. We print:

- each participant's bit `b`, the ±1 encoding `s ∈ {+1, N−1}`, the randomness `r`, and the puzzle `(u, v)` where `u = g^r mod N`, `v = h^r · s mod N`,
- the running products `ũ = ∏ uᵢ mod N` and `ṽ = ∏ vᵢ mod N` as `PEval` folds the puzzles,
- the `PSolve` chain: every squaring `w = w² mod N`, then `w⁻¹`, then `s = v · w⁻¹ mod N`,
- mapping `+1 → 0`, `N−1 → 1` to recover the XOR. -->

In [ ]:
import random

# ---------- 1. Setup (tiny parameters) -------------------------------------
random.seed(1)
demo_pp_m = mhtlp_setup(bits=16, T=5)
Nm, gm, hm, Tm = demo_pp_m['N'], demo_pp_m['g'], demo_pp_m['h'], demo_pp_m['T']
print('--- public parameters -----------------------------------------------')
print(f'N = {Nm}   ({Nm.bit_length()} bits)')
print(f'g = {gm}')
print(f'h = g^(2^T) mod N = {hm}')
print(f'T = {Tm}\n')

# ---------- 2. Four participants commit to bits ----------------------------
bits_truth = [1, 0, 1, 1]                        # ground-truth XOR = 1
expected_xor = 0
for b in bits_truth:
    expected_xor ^= b

def mhtlp_pgen_verbose(bit, who):
    s = 1 if bit == 0 else Nm - 1                # ±1 encoding
    r = random.randrange(1, Nm * Nm)
    u = pow(gm, r, Nm)
    blinding = pow(hm, r, Nm)                    # h^r — the mask
    v = (blinding * s) % Nm
    sign = '+1' if s == 1 else 'N-1 (≡ -1)'
    print(f'  {who}: bit={bit}  s={sign}  r={r}')
    print(f'         u = g^r mod N      = {u}')
    print(f'         h^r mod N (mask)   = {blinding}')
    print(f'         v = h^r · s mod N  = {v}')
    return (u, v)

puzzles = []
for i, b in enumerate(bits_truth, start=1):
    print(f'--- participant {i} ----------------------------------------------------')
    puzzles.append(mhtlp_pgen_verbose(b, f'P{i}'))
    print()

# ---------- 3. PEval — multiply the puzzles --------------------------------
print('--- PEval (multiplicative fold over Z*_N) ---------------------------')
u_tilde, v_tilde = 1, 1
for k, (u, v) in enumerate(puzzles, start=1):
    u_tilde = (u_tilde * u) % Nm
    v_tilde = (v_tilde * v) % Nm
    print(f'  after P{k}:   ũ = {u_tilde}   ṽ = {v_tilde}')
combined = (u_tilde, v_tilde)
print(f'  combined puzzle: {combined}\n')

# ---------- 4. PSolve step by step -----------------------------------------
print('--- PSolve on the combined puzzle -----------------------------------')
u, v = combined
w = u
print(f'  start: w = u = {w}')
for step in range(1, Tm + 1):
    w = pow(w, 2, Nm)
    print(f'  after squaring #{step}:  w = {w}')
w_inv = pow(w, -1, Nm)
s_recovered = (v * w_inv) % Nm
print(f'  w^-1 mod N         = {w_inv}')
print(f'  s = v · w^-1 mod N = {s_recovered}')

# ---------- 5. Map ±1 back to a bit ----------------------------------------
if s_recovered == 1:
    outcome = 0
    encoding = '+1  →  bit 0  (tails)'
elif s_recovered == Nm - 1:
    outcome = 1
    encoding = 'N-1 ≡ -1  →  bit 1  (heads)'
else:
    raise AssertionError('combined secret is not ±1 — algebra broke')
print(f'  decoded: {encoding}\n')

print('=== COIN-FLIP RESULT ================================================')
print(f'XOR ground truth = {bits_truth[0]} ⊕ {bits_truth[1]} ⊕ {bits_truth[2]} ⊕ {bits_truth[3]} = {expected_xor}')
print(f'PSolve outcome   = {outcome}')
assert outcome == expected_xor
print('coin lands on the homomorphic XOR of all four bits')

--- public parameters -----------------------------------------------
N = 1809778813   (31 bits)
g = 1444007172
h = g^(2^T) mod N = 79155629
T = 5

--- participant 1 ----------------------------------------------------
  P1: bit=1  s=N-1 (≡ -1)  r=1176298286020543698
         u = g^r mod N      = 1397543551
         h^r mod N (mask)   = 85352560
         v = h^r · s mod N  = 1724426253

--- participant 2 ----------------------------------------------------
  P2: bit=0  s=+1  r=2284791064310049674
         u = g^r mod N      = 906144970
         h^r mod N (mask)   = 16656428
         v = h^r · s mod N  = 16656428

--- participant 3 ----------------------------------------------------
  P3: bit=1  s=N-1 (≡ -1)  r=2072911645936348997
         u = g^r mod N      = 89991394
         h^r mod N (mask)   = 410665486
         v = h^r · s mod N  = 1399113327

--- participant 4 ----------------------------------------------------
  P4: bit=1  s=N-1 (≡ -1)  r=3004942216765706338
         u = g^r m